# Hyperparameter Tuning

Use GridSearchCV and RandomizedSearchCV to select hyperparameters with cross-validation.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import loguniform
data = load_breast_cancer(as_frame=True); X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {'model__C': [0.01, 0.1, 1, 10], 'model__solver': ['liblinear', 'lbfgs']}
search = GridSearchCV(pipeline, param_grid, scoring='f1_macro', cv=cv, n_jobs=-1)
search.fit(X_train, y_train)
print('Best parameters:', search.best_params_); print('Best CV macro-F1:', round(search.best_score_, 4))
print('Test accuracy:', round(accuracy_score(y_test, search.predict(X_test)), 4))
print(classification_report(y_test, search.predict(X_test), target_names=data.target_names))

Best parameters: {'model__C': 0.1, 'model__solver': 'liblinear'}
Best CV macro-F1: 0.9811
Test accuracy: 0.9825
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Inspect search results

In [2]:
results = pd.DataFrame(search.cv_results_)
results[['param_model__C', 'param_model__solver', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(10).round(4)

,param_model__C,param_model__solver,mean_test_score,std_test_score,rank_test_score
2,0.10,liblinear,0.9811,0.0058,1
3,0.10,lbfgs,0.9811,0.0058,2
4,1.00,liblinear,0.9789,0.0087,3
5,1.00,lbfgs,0.9764,0.0106,4
0,0.01,liblinear,0.9718,0.0120,5
6,10.00,liblinear,0.9598,0.0121,6
7,10.00,lbfgs,0.9598,0.0121,6
1,0.01,lbfgs,0.9466,0.0170,8


## Randomized search
RandomizedSearchCV samples a fixed number of parameter combinations and can be more efficient for large search spaces.

In [3]:
random_search = RandomizedSearchCV(pipeline, {'model__C': loguniform(1e-3, 1e2), 'model__solver': ['liblinear', 'lbfgs']}, n_iter=12, scoring='f1_macro', cv=cv, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)
print('Best parameters:', random_search.best_params_); print('Best CV macro-F1:', round(random_search.best_score_, 4))

Best parameters: {'model__C': np.float64(0.16949768237884735), 'model__solver': 'liblinear'}
Best CV macro-F1: 0.9811


## Key takeaways
Grid search evaluates every specified combination; randomized search samples combinations. Use cross-validation for selection and a held-out test set for final evaluation. Pipeline parameters use double-underscore notation, such as `model__C`.